# 🌟 **CAIO MASTER TEMPLATE — COLAB (v1.0)**

Bem-vindo ao notebook que vai centralizar:

### ✔️ Transcrição ultra rápida com Whisper (GPU)
### ✔️ `.txt` + `.srt` com minutagem
### ✔️ Inteligência Epstemia com GPT-4o
### ✔️ Insights profundos sobre valor, persona e histórias
### ✔️ Identificação de trechos virais
### ✔️ Exportações em `.txt`, `.json`, `.md`, `.csv`
### ✔️ Nada é sobrescrito — tudo é preservado

---
## ⚠️ Antes de começar:

1. Certifique-se de estar usando **GPU**:
   - Runtime → Change runtime type → GPU
2. Tenha sua **API Key da OpenAI** em mãos.

---

## 📌 Estrutura deste Notebook

0. 🔐 Inserir API Key (Segurança Total)
1. 📂 Montar Google Drive
2. 📤 Upload opcional de áudios
3. 🎙️ Transcrição (Whisper GPU medium + paralelismo)
4. 🛠️ Pós-processamento (opcional)
5. 🧠 Inteligência Epstemia (GPT-4o)
6. 🎨 Geração de Conteúdo
7. 📦 Exportação organizada

---

# 🔐 **Seção 0 — Inserir API Key (Seguro)**

Esta célula permite inserir sua chave da OpenAI com total segurança:

- nada é exibido na tela
- nada é armazenado em arquivo
- nada é salvo no notebook
- só existe na memória RAM enquanto o notebook está rodando

👉 Rode esta célula primeiro *sempre*.

In [ ]:
import getpass, os

print("🔐 Insira sua OpenAI API Key (não será exibida e não será salva):")
api_key = getpass.getpass()

if not api_key.startswith("sk-"):
    raise ValueError("❌ API Key inválida. Certifique-se de que começa com 'sk-'.")

os.environ["OPENAI_API_KEY"] = api_key

print("✔ API Key carregada com segurança!")

# 📂 **Seção 1 — Montar Google Drive**

O notebook salvará automaticamente:

```
ativos/ai lab/transcrições
ativos/ai lab/inteligência_gerada
```

Estas pastas serão criadas automaticamente se não existirem.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/ativos/ai lab"
TRANSCRICOES_DIR = f"{BASE_DIR}/transcrições"
INTEL_DIR = f"{BASE_DIR}/inteligência_gerada"

os.makedirs(TRANSCRICOES_DIR, exist_ok=True)
os.makedirs(INTEL_DIR, exist_ok=True)

print("📁 Pastas criadas/verificadas com sucesso:")
print(TRANSCRICOES_DIR)
print(INTEL_DIR)

# 📤 **Seção 2 — Upload Opcional de Áudios**

Você pode subir arquivos manualmente se quiser.

- Eles serão enviados direto para a pasta:
```
ativos/ai lab/transcrições
```
- Arquivos já existentes **não serão sobrescritos**.

In [ ]:
from google.colab import files
import shutil

print("📤 Selecione arquivos .mp3 para enviar:")
uploaded = files.upload()

for name in uploaded.keys():
    dest = f"{TRANSCRICOES_DIR}/{name}"
    if os.path.exists(dest):
        print(f"⚠ {name} já existe — pulando.")
    else:
        shutil.move(name, dest)
        print(f"✔ {name} salvo em {dest}")

# 🎙️ **Seção 3 — Transcrição (Whisper GPU + .txt + .srt)**

Esta célula faz a transcrição completa com GPU usando:
- Modelo **medium** por padrão
- Gera **.txt** e **.srt**
- Corre em **paralelo** (acelera MUITO)
- Não sobrescreve arquivos já existentes

Utiliza `whisper` (CLI) porque é mais rápido que usar Python.

In [ ]:
!pip install openai-whisper
!apt-get install ffmpeg -y

In [ ]:
import subprocess, glob, os, multiprocessing, time

MODEL = "medium"       # modelo padrão
PARALLEL = 4            # Colab aguenta 4 com folga

# listar arquivos na pasta
mp3_files = glob.glob(f"{TRANSCRICOES_DIR}/*.mp3")
total = len(mp3_files)

if total == 0:
    print("⚠ Nenhum arquivo .mp3 na pasta de transcrições.")
else:
    print(f"🎧 {total} áudios encontrados.")

def barra(i, total, largura=30):
    pct = i/total
    done = int(largura*pct)
    left = largura - done
    return f"[{'#'*done}{'-'*left}] {int(pct*100)}%"

def process_file(path):
    base = os.path.splitext(os.path.basename(path))[0]
    txt_out = f"{TRANSCRICOES_DIR}/{base}.txt"
    srt_out = f"{TRANSCRICOES_DIR}/{base}.srt"

    # evitar sobrescrever
    if os.path.exists(txt_out) and os.path.exists(srt_out):
        return f"⚠ {base}: já existe — pulando"

    subprocess.run([
        "whisper", path,
        "--model", MODEL,
        "--language", "pt",
        "--output_dir", TRANSCRICOES_DIR,
        "--output_format", "txt",
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # gerar SRT separadamente
    subprocess.run([
        "whisper", path,
        "--model", MODEL,
        "--language", "pt",
        "--output_dir", TRANSCRICOES_DIR,
        "--output_format", "srt",
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    return f"✔ {base}: transcrição concluída"

# processamento paralelo
inicio = time.time()

with multiprocessing.Pool(PARALLEL) as p:
    for i, msg in enumerate(p.imap_unordered(process_file, mp3_files), start=1):
        print(barra(i, total), msg, end="\r")

print("\n\n✨ Transcrições concluídas!")
print(f"⏱ Tempo total: {time.time() - inicio:.1f}s")

# 🛠️ **Seção 4 — Pós-processamento (opcional)**

Aqui podemos limpar texto, padronizar quebras de linha, remover ruídos, etc.

Você pode pular esta parte se quiser.

In [ ]:
# Exemplo simples de limpeza textual (opcional)
import glob

txt_files = glob.glob(f"{TRANSCRICOES_DIR}/*.txt")
print(f"Encontrados {len(txt_files)} arquivos .txt para limpeza opcional.")

def limpar_texto(texto):
    texto = texto.replace("  ", " ")
    texto = texto.replace("\n\n", "\n")
    return texto.strip()

for txt in txt_files:
    with open(txt, "r", encoding="utf-8") as f:
        conteudo = f.read()

    novo = limpar_texto(conteudo)

    with open(txt, "w", encoding="utf-8") as f:
        f.write(novo)

print("✔ Limpeza concluída!")

# 🧠 **Seção 5 — Inteligência Epstemia (GPT-4o)**

Esta é a parte mais importante do notebook.

Ela:

### ✔ Lê TODAS as transcrições da pasta
### ✔ Usa o modelo GPT-4o para análise profunda
### ✔ Extrai histórias reais
### ✔ Identifica Proposta de Valor Percebida
### ✔ Analisa Persona, Instintos, Jornada do Herói
### ✔ Gera insights para aumentar valor
### ✔ Gera estratégias de conexão
### ✔ Identifica Trechos com Potencial Viral
### ✔ Exporta tudo em: `.txt`, `.json`, `.md`, `.csv`

---

⚠️ **IMPORTANTE:**
- Esta seção usa GPT-4o.
- Certifique-se de ter rodado a Seção 0 para inserir a API key.
- Certifique-se de que há `.txt` na pasta de transcrições.

---

In [ ]:
import os, json, csv
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("✔ Conexão com GPT-4o pronta.")

## 📥 **5.1 — Carregar todas as transcrições**

Aqui vamos:
- ler todas as transcrições da pasta
- unir tudo em um buffer gigante
- preparar para análise profunda

In [ ]:
txt_files = sorted(glob.glob(f"{TRANSCRICOES_DIR}/*.txt"))
print(f"📚 Encontrados {len(txt_files)} arquivos de transcrição.")

transcricoes = {}
buffer_total = ""

for path in txt_files:
    nome = os.path.basename(path)
    with open(path, "r", encoding="utf-8") as f:
        conteudo = f.read().strip()
    transcricoes[nome] = conteudo
    buffer_total += f"\n\n=== {nome} ===\n" + conteudo

print("✔ Transcrições carregadas e unificadas!")

## 🧬 **5.2 — Função Auxiliar para Salvar Outputs**

Esta função salva resultados em:
- `.txt`
- `.json`
- `.md`
- `.csv`

Tudo dentro da pasta `inteligência_gerada`.

In [ ]:
def salvar_output(nome_pasta, nome_arquivo, conteudo):
    pasta = f"{INTEL_DIR}/{nome_pasta}"
    os.makedirs(pasta, exist_ok=True)

    # TXT
    with open(f"{pasta}/{nome_arquivo}.txt", "w", encoding="utf-8") as f:
        f.write(conteudo)

    # MD
    with open(f"{pasta}/{nome_arquivo}.md", "w", encoding="utf-8") as f:
        f.write("# " + nome_arquivo + "\n\n" + conteudo)

    # JSON
    with open(f"{pasta}/{nome_arquivo}.json", "w", encoding="utf-8") as f:
        json.dump({"conteudo": conteudo}, f, ensure_ascii=False, indent=2)

    # CSV (uma coluna)
    with open(f"{pasta}/{nome_arquivo}.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["conteudo"])
        writer.writerow([conteudo])

    print(f"✔ Salvo em {pasta}/{nome_arquivo}.*")

## 🔥 **5.3 — Identificação de Histórias (casos para conteúdo e vendas)**

O modelo vai identificar:
- histórias reais
- episódios emocionais
- conflitos
- momentos de transformação
- metáforas vividas
- trechos que viram cortes
- trechos que conectam profundamente com a audiência
- narrativas que podem ser usadas para vendas (de forma ética)


In [ ]:
prompt_historias = f"""
Você é uma inteligência analítica especializada em narrativas, psicologia profunda, hermetismo aplicado, storytelling e análise de transformação humana.

A seguir está um conjunto de transcrições reais de encontros da Epstemia.

Sua tarefa:
1. Identificar HISTÓRIAS PESSOAIS relevantes.
2. Identificar momentos de tensão → clareza.
3. Identificar episódios que podem virar conteúdo.
4. Identificar histórias que podem ser usadas como referência para vendas.
5. Identificar histórias que revelam conflitos profundos.
6. Identificar metáforas emocionais fortes.
7. Destacar trechos com força de narrativa.

Escreva de forma ULTRA clara, estruturada e direta.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_historias}]
)

resultado_hist = resp.choices[0].message.content
salvar_output("historias", "historias_identificadas", resultado_hist)

## 💎 **5.4 — Proposta de Valor Percebida**

Aqui entendemos o que as pessoas sentem que a Epstemia realmente entrega.

O que elas acreditam que podem transformar ou conquistar por meio do trabalho.

In [ ]:
prompt_valor = f"""
Analise profundamente as transcrições e extraia:

1. QUAL É A PROPOSTA DE VALOR PERCEBIDA da Epstemia.
2. O que as pessoas acreditam que vão mudar.
3. O que elas esperam viver.
4. Que dores esperam resolver.
5. Como elas descrevem o trabalho do Caio.
6. Palavras e frases usadas espontaneamente para falar da transformação.

Organize em forma ultra clara, objetiva e estratégica.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_valor}]
)

resultado_valor = resp.choices[0].message.content
salvar_output("proposta_de_valor", "proposta_de_valor_percebida", resultado_valor)

## 💡 **5.5 — Insights sobre COMO GERAR MAIS VALOR**

Aqui o sistema identifica oportunidades reais para ampliar impacto e profundidade da Epstemia.

In [ ]:
prompt_valor2 = f"""
Com base nas transcrições, responda:

1. Como a Epstemia pode gerar ainda mais valor?
2. O que está faltando na vida emocional dessas pessoas?
3. Quais lacunas de clareza existem?
4. Quais oportunidades de aprofundamento surgem?
5. Quais temas geram mais impacto nelas?

Entregue respostas diretas, aplicáveis e estratégicas.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_valor2}]
)

resultado_valor2 = resp.choices[0].message.content
salvar_output("insights_de_valor", "como_gerar_mais_valor", resultado_valor2)

## 🧬 **5.6 — Persona, Eneagrama, Instinto e Jornada do Herói**

Nesta análise o sistema identifica:
- quem é a pessoa
- dores centrais
- medos
- desejos
- tipo (eneagrama)
- instinto dominante
- fase atual da jornada do herói
- necessidades emocionais


In [ ]:
prompt_persona = f"""
Com base nas transcrições, identifique:

1. Quem é essa pessoa?
2. O que ela está buscando?
3. Qual é a dor central dela?
4. Qual é o medo dela?
5. Qual é o desejo profundo dela?
6. Possível tipo (Eneagrama)
7. Instinto predominante (SP/SX/SO)
8. Fase da Jornada do Herói (Joseph Campbell)
9. Psicodinâmica interna

Escreva de forma profunda, humana e clara.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_persona}]
)

resultado_persona = resp.choices[0].message.content
salvar_output("persona", "analise_persona_completa", resultado_persona)

## 🔗 **5.7 — Estratégias de Conexão e Expansão da Audiência**

O sistema responde:
- como encontrar mais pessoas iguais
- como gerar identificação profunda
- quando essa pessoa lembra da Epstemia
- quais são os melhores temas
- como aumentar conexão emocional


In [ ]:
prompt_conn = f"""
Com base nas transcrições:
1. Como encontrar mais pessoas como essas?
2. Quando elas lembram da Epstemia?
3. Quais são os gatilhos emocionais?
4. Como aumentar conexão e profundidade?
5. Quais formatos de conteúdo funcionam melhor?

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_conn}]
)

resultado_conn = resp.choices[0].message.content
salvar_output("conexao", "estrategias_de_conexao", resultado_conn)

## 🎬 **5.8 — Trechos com Alto Potencial Viral**

Aqui o sistema identifica:
- punchlines
- tensões emocionais
- frases universalmente identificáveis
- momentos de insight
- trechos perfeitos para reels
- timecodes (via `.srt`)


In [ ]:
prompt_viral = f"""
Analise as transcrições e identifique:
1. Trechos com forte potencial viral.
2. Frases com punchline.
3. Momentos que geram identificação universal.
4. Momentos de conflito → insight.
5. Trechos emocionalmente potentes.
6. Escreva com timecodes aproximados.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_viral}]
)

resultado_viral = resp.choices[0].message.content
salvar_output("trechos_virais", "trechos_com_potencial_viral", resultado_viral)

# 🎨 **Seção 6 — Geração de Conteúdo (Carrosséis, Reels, Textos)**

Com base em TODAS as análises anteriores, aqui o GPT-4o gera conteúdo pronto para:

- Stories
- Carrosséis
- Reels
- Texto longo
- Post profundo (Mago)
- Post direto (Fora da Lei)
- Versão exploradora (Explorador)
- Roteiro para vídeo
- CTA contextualizado

Tudo usando o estilo e profundidade que caracterizam o Caio / Epstemia.

In [ ]:
prompt_conteudo = f"""
Com base nas transcrições e NAS ANÁLISES previamente feitas (sem precisar que eu envie tudo novamente), gere os seguintes conteúdos:

1. **Carrossel**: 7 a 10 slides, profundos e práticos.
2. **Roteiro de Reels**: 20 a 40 segundos.
3. **Texto Longo**: estilo Epstemia.
4. **Post Profundo (Mago)**.
5. **Post Direto (Fora da Lei)**.
6. **Versão Contemplativa (Explorador)**.
7. **Mini-Aula (1 min)**.
8. **CTA emocional + CTA racional**.

Use o estilo literário, filosófico e pedagógico presente no trabalho do Caio.
Use linguagem clara, sensível e realista.
Contexto: Epstemia, Algoritmo da Leveza, Clã dos Sutis.

TRANSCRIÇÕES:
{buffer_total}
"""

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt_conteudo}]
)

resultado_conteudo = resp.choices[0].message.content
salvar_output("conteudo", "conteudo_criado", resultado_conteudo)

# 📦 **Seção 7 — Exportação Final**

Nesta célula você verá onde tudo foi salvo, organizado em:

```
ativos/ai lab/inteligência_gerada/
    historias/
    proposta_de_valor/
    insights_de_valor/
    persona/
    conexao/
    trechos_virais/
    conteudo/
```

Cada pasta contém:
- `.txt`
- `.md`
- `.json`
- `.csv`

---
Tudo fica salvo **direto no seu Google Drive**.

In [ ]:
print("📁 Resultados salvos na pasta:")
print(INTEL_DIR)

print("\n✨ Processo completo!")
print("Você agora tem uma análise profunda da sua comunidade e conteúdo pronto para criação.")